# 05. Feature Selection with Extra Trees

**Research question:** Does selecting voice features reduce the mean validation MAE of Extra Trees compared with using all 16 features?

**Comparison:** Use the same three patient-wise folds and Extra Trees settings as Notebook 04. Select features using only the training patients in each fold.

## 1. Import Libraries

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

## 2. Load the Parkinson’s Telemonitoring Dataset

In [2]:
data_path = Path("../data/parkinsons_updrs.data")
df = pd.read_csv(data_path)

print("Recordings:", len(df))
print("Columns:", df.shape[1])
print("Unique patients:", df["subject#"].nunique())

Recordings: 5875
Columns: 22
Unique patients: 42


## 3. Define Voice Features, Target, and Patient Groups

In [3]:
voice_features = [
    "Jitter(%)", "Jitter(Abs)", "Jitter:RAP", "Jitter:PPQ5", "Jitter:DDP",
    "Shimmer", "Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5",
    "Shimmer:APQ11", "Shimmer:DDA", "NHR", "HNR", "RPDE", "DFA", "PPE"
]

X = df[voice_features]
y = df["motor_UPDRS"]
groups = df["subject#"]

print("Input shape (recordings, features):", X.shape)
print("Target:", y.name)
print("Patient ID column:", groups.name)

Input shape (recordings, features): (5875, 16)
Target: motor_UPDRS
Patient ID column: subject#


## 4. Recreate and Verify the Patient-wise Train/Test Split

In [4]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
groups_train = groups.iloc[train_idx]

train_ids = set(groups_train)
test_ids = set(groups.iloc[test_idx])
previous_test_ids = {5, 9, 14, 20, 26, 27, 30, 31, 40}

print("Training patients:", len(train_ids))
print("Test patients:", len(test_ids))
print("Patient overlap:", len(train_ids & test_ids))
print("Same test patients as Notebook 04:", test_ids == previous_test_ids)

Training patients: 33
Test patients: 9
Patient overlap: 0
Same test patients as Notebook 04: True


## 5. Recreate the Three Patient-wise Validation Folds

In [5]:
cv = GroupKFold(n_splits=3)
folds = list(cv.split(X_train, y_train, groups=groups_train))

for fold, (fit_idx, val_idx) in enumerate(folds, start=1):
    fit_ids = set(groups_train.iloc[fit_idx])
    val_ids = set(groups_train.iloc[val_idx])

    print(f"Fold {fold}: training patients = {len(fit_ids)}, "
          f"validation patients = {len(val_ids)}, "
          f"overlap = {len(fit_ids & val_ids)}")

Fold 1: training patients = 22, validation patients = 11, overlap = 0
Fold 2: training patients = 22, validation patients = 11, overlap = 0
Fold 3: training patients = 22, validation patients = 11, overlap = 0


## 6. Inspect Feature Correlations in Fold 1 Training Data

In [6]:
fit_idx, val_idx = folds[0]
X_fit = X_train.iloc[fit_idx]

feature_corr = X_fit.corr(method="pearson")

jitter_cols = ["Jitter(%)", "Jitter(Abs)", "Jitter:RAP"]
feature_corr.loc[jitter_cols, jitter_cols].round(3)

,Jitter(%),Jitter(Abs),Jitter:RAP
Jitter(%),1.000,0.923,0.985
Jitter(Abs),0.923,1.000,0.909
Jitter:RAP,0.985,0.909,1.000


**Observation:** Among the three jitter features examined in the Fold 1 training data, Jitter(%) and Jitter:RAP had the highest Pearson correlation (r = 0.985). This suggests that they capture similar variation, but it does not establish which feature is more informative for predicting motor UPDRS.

## 7. Group Voice Features by Their Fold 1 Training Correlations

In [7]:
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)

# Each row represents one voice feature, not one patient or recording.
cluster_ids = kmeans.fit_predict(feature_corr)

feature_groups = pd.DataFrame({
    "feature": feature_corr.index,
    "cluster": cluster_ids + 1
})

feature_groups.groupby("cluster")["feature"].agg(list)

C:\Users\mamona\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


cluster
1                                          [RPDE, PPE]
2                                                [HNR]
3                                                [DFA]
4    [Jitter(%), Jitter(Abs), Jitter:RAP, Jitter:PP...
Name: feature, dtype: object

In [8]:
for cluster, rows in feature_groups.groupby("cluster"):
    names = rows["feature"].tolist()
    print(f"Group {cluster} ({len(names)} features): {', '.join(names)}")

Group 1 (2 features): RPDE, PPE
Group 2 (1 features): HNR
Group 3 (1 features): DFA
Group 4 (12 features): Jitter(%), Jitter(Abs), Jitter:RAP, Jitter:PPQ5, Jitter:DDP, Shimmer, Shimmer(dB), Shimmer:APQ3, Shimmer:APQ5, Shimmer:APQ11, Shimmer:DDA, NHR


## 8. Select One Representative per Cluster Using Mutual Information

In [9]:
y_fit = y_train.iloc[fit_idx]

mi_by_feature = pd.Series(
    mutual_info_regression(X_fit, y_fit, random_state=42),
    index=X_fit.columns
)

ranked = feature_groups.copy()
ranked["MI_score"] = ranked["feature"].map(mi_by_feature)

selected_fold1 = (
    ranked.sort_values(["cluster", "MI_score"], ascending=[True, False])
    .drop_duplicates("cluster")
    .reset_index(drop=True)
)

selected_fold1.round(4)

,feature,cluster,MI_score
0,PPE,1,0.1462
1,HNR,2,0.1917
2,DFA,3,0.2693
3,Shimmer:APQ11,4,0.1603


## 9. Fit Extra Trees Using the Four Selected Features in Fold 1

In [10]:
selected_features = selected_fold1["feature"].tolist()
X_fit_selected = X_fit[selected_features]

et_selected_fold1 = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42
)

et_selected_fold1.fit(X_fit_selected, y_fit)

print("Selected features:", selected_features)
print("Training shape (recordings, features):", X_fit_selected.shape)

Selected features: ['PPE', 'HNR', 'DFA', 'Shimmer:APQ11']
Training shape (recordings, features): (3091, 4)


## 10. Evaluate Extra Trees with Selected Features on Fold 1

In [12]:
X_val_selected = X_train.iloc[val_idx][selected_features]
y_val = y_train.iloc[val_idx]

pred_selected = et_selected_fold1.predict(X_val_selected)
mae_selected_fold1 = mean_absolute_error(y_val, pred_selected)

print(f"Fold 1 MAE with 4 selected features: {mae_selected_fold1:.4f}")

Fold 1 MAE with 4 selected features: 8.0865


**Observation:** On the same Fold 1 validation recordings, reducing the input from 16 voice features to four selected features increased the Extra Trees MAE from 7.5672 to 8.0865 points. Feature selection did not improve prediction on this fold.

## 11. Evaluate Feature Selection Across All Three Patient-wise Folds

In [13]:
selection_results = []

for fold, (fit_idx, val_idx) in enumerate(folds, start=1):
    X_fit, X_val = X_train.iloc[fit_idx], X_train.iloc[val_idx]
    y_fit, y_val = y_train.iloc[fit_idx], y_train.iloc[val_idx]

    # Group features using training recordings only.
    corr = X_fit.corr()
    clusters = KMeans(
        n_clusters=4, random_state=42, n_init=20
    ).fit_predict(corr)

    # Select one feature per group using training targets only.
    scores = pd.DataFrame({
        "feature": X_fit.columns,
        "cluster": clusters,
        "MI_score": mutual_info_regression(X_fit, y_fit, random_state=42)
    })

    chosen = (
        scores.sort_values(["cluster", "MI_score"], ascending=[True, False])
        .drop_duplicates("cluster")["feature"]
        .tolist()
    )

    model = ExtraTreesRegressor(n_estimators=100, random_state=42)
    model.fit(X_fit[chosen], y_fit)
    predictions = model.predict(X_val[chosen])

    selection_results.append({
        "fold": fold,
        "selected_features": ", ".join(chosen),
        "selected_MAE": mean_absolute_error(y_val, predictions)
    })

selection_cv = pd.DataFrame(selection_results)
print(selection_cv.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

C:\Users\mamona\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\mamona\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\mamona\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


 fold                   selected_features  selected_MAE
    1        PPE, HNR, DFA, Shimmer:APQ11        8.0865
    2 Jitter:PPQ5, HNR, DFA, Shimmer:APQ5        8.6399
    3 Shimmer:APQ5, HNR, Jitter(Abs), DFA        8.0324


**Observation:** Selecting four voice features reduced the mean Extra Trees MAE from 8.2712 to 8.2529 points across three patient-wise folds. MAE improved only in Fold 3, so the improvement was not consistent across folds. The training-mean baseline retained the lowest mean MAE (7.2706 points).

## 12. Save the Fold-wise Feature Selection Comparison

In [14]:
results_dir = Path("../results")

all_features = (
    pd.read_csv(results_dir / "patient_wise_model_comparison.csv")
    [["fold", "extra_trees_MAE"]]
    .rename(columns={"extra_trees_MAE": "all_16_MAE"})
)

comparison = all_features.merge(selection_cv, on="fold", validate="one_to_one")
comparison["MAE_change"] = comparison["selected_MAE"] - comparison["all_16_MAE"]

comparison.to_csv(results_dir / "feature_selection_fold_comparison.csv", index=False)
print(comparison.round(4).to_string(index=False))

 fold  all_16_MAE                   selected_features  selected_MAE  MAE_change
    1      7.5672        PPE, HNR, DFA, Shimmer:APQ11        8.0865      0.5193
    2      8.6373 Jitter:PPQ5, HNR, DFA, Shimmer:APQ5        8.6399      0.0026
    3      8.6090 Shimmer:APQ5, HNR, Jitter(Abs), DFA        8.0324     -0.5766
